# API-Sports v3 全面数据采集

> 目标：从 api-football.com 采集尽可能全面的数据，用于 2026 世界杯冠军预测模型

## 采集约束
- **免费版**：100 次/天，10 次/分钟
- 每个 Cell 可独立运行，带**断点续传**（缓存已采集数据，跳过已有）
- 按天运行不同 Cell，避免超限

## 采集调度

| 天次 | 运行 Cell | 预估请求 | 任务 |
|------|-----------|----------|------|
| Day 1 | Cell 2 + Cell 3 | ~50 | 补全球队 ID + 2022 WC 统计 |
| Day 2 | Cell 2(续) + Cell 4 | ~50 | 补全 ID + 2018 WC |
| Day 3 | Cell 5 + Cell 9 | ~30 | 近期大赛 + 2026 赛程 |
| Day 4-6 | Cell 6 (分批) | ~96 | 近期比赛 |
| Day 7-8 | Cell 7 (分批) | ~48 | 球员数据 |
| Day 9-10 | Cell 8 (分批) | ~72 | H2H 矩阵 |

## 1. API 配置与工具函数

In [1]:
import requests
import json
import time
import os
from pathlib import Path
from datetime import datetime

# ============ API 配置 ============
API_KEY = 'f1d6df65ff4f697f9a5b56dfb8824cfe'
HEADERS = {'x-apisports-key': API_KEY}
BASE_URL = 'https://v3.football.api-sports.io'
DATA_DIR = Path('wc2026_raw_data')

# 免费版限制
DAILY_LIMIT = 100
RATE_LIMIT_INTERVAL = 6.1  # 秒，10次/分钟 -> 每6秒1次

# ============ 全局请求计数器 ============
request_count = 0

def reset_counter():
    global request_count
    request_count = 0

def get_remaining():
    return DAILY_LIMIT - request_count

def check_quota(needed=1):
    """检查剩余配额，不足时警告"""
    if get_remaining() < needed:
        print(f"[!] 配额不足: 还需 {needed} 次，仅剩 {get_remaining()} 次。请明天继续。")
        return False
    return True

# ============ 核心请求函数 ============
def api_get(endpoint, params=None, use_cache=True, cache_file=None, sleep=True):
    """
    带速率控制、配额检查、缓存的 API 请求
    
    Parameters:
        endpoint: API 端点 (如 'fixtures', 'teams')
        params: 请求参数字典
        use_cache: 是否使用缓存（断点续传）
        cache_file: 缓存文件名（默认按 endpoint+params 自动生成）
        sleep: 是否等待（关闭可加速，仅在确定不超限时使用）
    Returns:
        API 响应的 response 列表，或 None（配额不足/出错）
    """
    global request_count
    
    if params is None:
        params = {}
    
    # 自动生成缓存文件名
    if use_cache and cache_file is None:
        safe_params = '_'.join(f"{k}={v}" for k, v in sorted(params.items()))
        cache_file = f"cache_{endpoint}_{safe_params}.json"
    
    # 检查缓存
    if use_cache and cache_file:
        cache_path = DATA_DIR / cache_file
        if cache_path.exists():
            mtime = datetime.fromtimestamp(cache_path.stat().st_mtime)
            print(f"  [缓存命中] {cache_file} (更新于 {mtime.strftime('%m-%d %H:%M')})")
            with open(cache_path) as f:
                return json.load(f)
    
    # 配额检查
    if not check_quota():
        return None
    
    # 速率控制
    if sleep and request_count > 0:
        time.sleep(RATE_LIMIT_INTERVAL)
    
    # 发起请求
    try:
        resp = requests.get(f"{BASE_URL}/{endpoint}", headers=HEADERS, params=params, timeout=15)
        request_count += 1
        data = resp.json()
        
        if data.get('errors') and len(data['errors']) > 0:
            error_msg = list(data['errors'].values())[0] if isinstance(data['errors'], dict) else str(data['errors'])
            if 'rate limit' in str(error_msg).lower():
                print(f"  [速率限制] 等待 60 秒...")
                time.sleep(60)
                resp = requests.get(f"{BASE_URL}/{endpoint}", headers=HEADERS, params=params, timeout=15)
                request_count += 1
                data = resp.json()
            else:
                print(f"  [API 错误] {endpoint}: {error_msg}")
                return None
        
        results = data.get('response', [])
        
        # 写入缓存
        if use_cache and cache_file and results:
            os.makedirs(DATA_DIR, exist_ok=True)
            with open(DATA_DIR / cache_file, 'w') as f:
                json.dump(results, f, indent=2, ensure_ascii=False)
            print(f"  [已缓存] {cache_file} ({len(results)} 条记录)")
        
        print(f"  [请求 {request_count}/{DAILY_LIMIT}] GET /{endpoint} -> {len(results)} 条")
        return results
        
    except Exception as e:
        print(f"  [请求失败] {e}")
        return None

def api_get_all_pages(endpoint, params=None, use_cache=True, cache_file=None, max_pages=10):
    """自动翻页获取全量数据"""
    if params is None:
        params = {}
    
    all_results = []
    
    # 生成主缓存文件名（不含 page）
    if cache_file is None:
        safe_params = '_'.join(f"{k}={v}" for k, v in sorted(params.items()))
        cache_file = f"cache_{endpoint}_{safe_params}"
    
    for page in range(1, max_pages + 1):
        page_cache = f"{cache_file}_p{page}.json" if page > 1 else f"{cache_file}.json"
        
        # 检查是否已有该页缓存
        if use_cache:
            cache_path = DATA_DIR / page_cache
            if cache_path.exists():
                with open(cache_path) as f:
                    page_data = json.load(f)
                if not page_data:  # 空页 = 没有更多数据
                    break
                all_results.extend(page_data)
                print(f"  [缓存] Page {page}: {len(page_data)} 条")
                continue
        
        # 请求该页
        page_params = {**params, 'page': page}
        page_data = api_get(endpoint, params=page_params, use_cache=False, sleep=(page > 1))
        
        if page_data is None:
            # API 出错，不写空缓存，尝试不带 page 参数重试一次
            if page == 1:
                print(f"  [重试] 不带 page 参数请求 {endpoint}...")
                page_data = api_get(endpoint, params=params, use_cache=False, sleep=False)
                if page_data is not None:
                    # 不支持分页的端点，一次返回全部
                    all_results.extend(page_data)
                    if cache_file:
                        os.makedirs(DATA_DIR, exist_ok=True)
                        with open(DATA_DIR / page_cache, 'w') as f:
                            json.dump(page_data, f, indent=2, ensure_ascii=False)
                    break
            else:
                # 重试也失败，停止翻页
                break
        elif page_data is not None and len(page_data) == 0:
            # 真正的空结果（数据结束）
            with open(DATA_DIR / page_cache, 'w') as f:
                json.dump([], f)
            break
        
        all_results.extend(page_data)
        
        # 缓存该页
        os.makedirs(DATA_DIR, exist_ok=True)
        with open(DATA_DIR / page_cache, 'w') as f:
            json.dump(page_data, f, indent=2, ensure_ascii=False)
    
    # 保存合并后的完整数据
    if all_results and cache_file:
        merged_file = f"{cache_file}_all.json"
        with open(DATA_DIR / merged_file, 'w') as f:
            json.dump(all_results, f, indent=2, ensure_ascii=False)
        print(f"  [合并] 共 {len(all_results)} 条 -> {merged_file}")
    
    return all_results

def save_data(filename, data):
    """保存数据到 JSON 文件"""
    os.makedirs(DATA_DIR, exist_ok=True)
    with open(DATA_DIR / filename, 'w') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"  [保存] {filename}")

def load_data(filename):
    """加载数据"""
    path = DATA_DIR / filename
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return None

# ============ 初始化 ============
reset_counter()
print("工具函数加载完成")
print(f"数据目录: {DATA_DIR.absolute()}")
print(f"已有文件: {len(list(DATA_DIR.glob('*.json')))} 个 JSON 文件")

工具函数加载完成
数据目录: /Users/user/worldcup_data/wc2026_raw_data
已有文件: 13 个 JSON 文件


## 2. 球队 ID 映射构建

从已有的 2026 WC 分组数据中提取 48 支球队名称，通过 API 搜索补全所有 API-Sports team ID。

> **预估请求**：~48 次（可分批运行，通过 `skip_to` 跳过已完成的队）

In [2]:
# === 2.1 提取 2026 世界杯全部参赛球队 ===
from collections import defaultdict

# 从已有的 team_*.json 文件提取球队名
teams_2026 = {}
for f in sorted(DATA_DIR.glob('team_*.json')):
    data = load_data(f.name)
    if data and data.get('team'):
        name = data['team']['name']
        teams_2026[name] = data['team']

# 加载已有的 API-Sports team ID
existing_ids = load_data('../api_sports_team_ids.json') or {}

print(f"2026 参赛队: {len(teams_2026)} 支")
print(f"已有 API-Sports ID: {len(existing_ids)} 支")
print(f"需要补全: {len(teams_2026) - len(set(teams_2026.keys()) & set(existing_ids.keys()))} 支")

# 展示分组
groups = defaultdict(list)
for f in sorted(DATA_DIR.glob('team_*.json')):
    data = load_data(f.name)
    if data and data.get('matches'):
        name = data['team']['name']
        group = data['matches'][0].get('group', 'Unknown')
        groups[group].append(name)

print("\n=== 2026 世界杯分组 ===")
for g, teams in sorted(groups.items()):
    print(f"  {g}: {', '.join(sorted(teams))}")

2026 参赛队: 0 支
已有 API-Sports ID: 49 支
需要补全: 0 支

=== 2026 世界杯分组 ===


In [24]:
# === 2.2 批量搜索球队 API-Sports ID ===
# 名称映射（2026 数据名 -> API-Sports 可能的名称）
NAME_ALIASES = {
    'United States': 'USA',
    'South Korea': 'South Korea',
    'Curaçao': 'Curacao',
    'Ivory Coast': 'Ivory Coast',
    'Cape Verde Islands': 'Cape Verde',
}

# 加载已有缓存
team_id_map = dict(existing_ids)  # 从已有 32 支开始

# 需要查找的队（排除已有的）
teams_to_find = []
for name in teams_2026:
    if name not in team_id_map:
        alias = NAME_ALIASES.get(name)
        if alias and alias in team_id_map:
            team_id_map[name] = team_id_map[alias]
            print(f"  [别名映射] {name} -> {alias} (id={team_id_map[alias]})")
        else:
            teams_to_find.append(name)

print(f"\n需要 API 查找: {len(teams_to_find)} 支")
print(f"剩余配额: {get_remaining()}")

# ============ 可调节的批次控制 ============
# 如果配额不够，设置 skip_to 跳到指定位置继续
skip_to = 0  # 修改此值可从第 N 支队开始
# ==========================================

for i, team_name in enumerate(teams_to_find):
    if i < skip_to:
        continue
    
    if not check_quota():
        print(f"\n[!] 在第 {i}/{len(teams_to_find)} 支队处停止，明天从 skip_to={i} 继续")
        break
    
    search_name = NAME_ALIASES.get(team_name, team_name)
    print(f"\n  [{i+1}/{len(teams_to_find)}] 搜索: {team_name} (search={search_name})")
    
    result = api_get('teams', params={'search': search_name}, cache_file=f"id_lookup_{team_name}.json")
    
    if result:
        # 找到匹配的国家队
        found = False
        for team in result:
            t = team['team']
            if t.get('national') and t['name'].lower() in search_name.lower():
                team_id_map[team_name] = t['id']
                print(f"    -> 找到: {t['name']} (id={t['id']})")
                found = True
                break
        
        if not found:
            # 取第一个结果
            t = result[0]['team']
            team_id_map[team_name] = t['id']
            print(f"    -> 使用: {t['name']} (id={t['id']}, national={t.get('national')})")
    else:
        print(f"    -> 未找到，跳过")

# 保存完整映射
save_data('../api_sports_team_ids.json', team_id_map)
print(f"\n=== 球队 ID 映射完成: {len(team_id_map)} 支 ===")

# 显示完整映射
for name, tid in sorted(team_id_map.items()):
    status = "[已有]" if name in existing_ids else "[新]"
    print(f"  {status} {name}: {tid}")


需要 API 查找: 0 支
剩余配额: 98
  [保存] ../api_sports_team_ids.json

=== 球队 ID 映射完成: 49 支 ===
  [已有] Algeria: 1532
  [已有] Argentina: 26
  [已有] Australia: 20
  [已有] Austria: 775
  [已有] Belgium: 1
  [已有] Brazil: 6
  [已有] Cameroon: 1530
  [已有] Canada: 5529
  [已有] Cape Verde Islands: 1533
  [已有] Colombia: 8
  [已有] Costa Rica: 29
  [已有] Croatia: 3
  [已有] Curaçao: 5530
  [已有] Denmark: 21
  [已有] Ecuador: 2382
  [已有] Egypt: 32
  [已有] England: 10
  [已有] France: 2
  [已有] Germany: 25
  [已有] Ghana: 1504
  [已有] Haiti: 2386
  [已有] Iran: 22
  [已有] Ivory Coast: 1501
  [已有] Japan: 12
  [已有] Jordan: 1548
  [已有] Mexico: 16
  [已有] Morocco: 31
  [已有] Netherlands: 1118
  [已有] New Zealand: 4673
  [已有] Norway: 1090
  [已有] Panama: 11
  [已有] Paraguay: 2380
  [已有] Poland: 24
  [已有] Portugal: 27
  [已有] Qatar: 1569
  [已有] Saudi Arabia: 23
  [已有] Scotland: 1108
  [已有] Senegal: 13
  [已有] Serbia: 14
  [已有] South Africa: 1531
  [已有] South Korea: 17
  [已有] Spain: 9
  [已有] Switzerland: 15
  [已有] Tunisia: 28
  [已有] USA: 2384
  [

## 3. 2022 世界杯完整数据补全

已采集 8 强球队统计，补全剩余 24 支 2022 参赛队的 `teams/statistics` 数据。

> **预估请求**：~24 次

In [25]:
# === 3.1 获取 2022 WC 全部 32 支球队列表 ===
# 先获取 2022 WC 参赛队（如果缓存不存在）
wc2022_teams = api_get_all_pages(
    'teams', params={'league': 1, 'season': 2022},
    cache_file='cache_wc2022_teams'
)
print(f"\n2022 WC 参赛队: {len(wc2022_teams)} 支")

# 构建 2022 球队 ID 列表
wc2022_team_ids = {t['team']['id']: t['team']['name'] for t in wc2022_teams}

# 已有统计的 8 强 ID（来自之前的采集）
existing_stats = load_data('wc2022_top8_team_stats.json') or {}
existing_stat_ids = set(existing_stats.keys())

# 需要补全的队
missing_stat_ids = set(wc2022_team_ids.keys()) - existing_stat_ids
print(f"已有统计: {len(existing_stat_ids)} 支")
print(f"需要补全: {len(missing_stat_ids)} 支")
print(f"剩余配额: {get_remaining()}")

  [API 错误] teams: The Page field do not exist.
  [重试] 不带 page 参数请求 teams...
  [请求 4/100] GET /teams -> 32 条
  [合并] 共 32 条 -> cache_wc2022_teams_all.json

2022 WC 参赛队: 32 支
已有统计: 0 支
需要补全: 32 支
剩余配额: 96


In [26]:
# === 3.2 补全 2022 WC 所有球队统计 ===
all_wc2022_stats = dict(existing_stats)  # 从已有 8 强开始

for tid in sorted(missing_stat_ids):
    team_name = wc2022_team_ids[tid]
    cache_file = f"cache_wc2022_teamstats_{tid}.json"
    
    # 检查缓存
    cached = load_data(cache_file)
    if cached:
        all_wc2022_stats[tid] = cached
        print(f"  [缓存] {team_name} (id={tid})")
        continue
    
    if not check_quota():
        print(f"\n[!] 配额不足，停止。已采集 {len(all_wc2022_stats)}/{len(wc2022_team_ids)} 支")
        break
    
    print(f"\n  采集 {team_name} (id={tid})...")
    result = api_get('teams/statistics', params={
        'league': 1, 'season': 2022, 'team': tid
    }, cache_file=cache_file)
    
    if result:
        all_wc2022_stats[tid] = result
    else:
        print(f"    -> 无数据，跳过")

# 保存完整统计
save_data('wc2022_all_team_stats.json', all_wc2022_stats)
print(f"\n=== 2022 WC 球队统计: {len(all_wc2022_stats)}/32 支 ===")


  采集 Belgium (id=1)...
  [已缓存] cache_wc2022_teamstats_1.json (11 条记录)
  [请求 5/100] GET /teams/statistics -> 11 条

  采集 France (id=2)...
  [已缓存] cache_wc2022_teamstats_2.json (11 条记录)
  [请求 6/100] GET /teams/statistics -> 11 条

  采集 Croatia (id=3)...
  [已缓存] cache_wc2022_teamstats_3.json (11 条记录)
  [请求 7/100] GET /teams/statistics -> 11 条

  采集 Brazil (id=6)...
  [已缓存] cache_wc2022_teamstats_6.json (11 条记录)
  [请求 8/100] GET /teams/statistics -> 11 条

  采集 Uruguay (id=7)...
  [已缓存] cache_wc2022_teamstats_7.json (11 条记录)
  [请求 9/100] GET /teams/statistics -> 11 条

  采集 Spain (id=9)...
  [已缓存] cache_wc2022_teamstats_9.json (11 条记录)
  [请求 10/100] GET /teams/statistics -> 11 条

  采集 England (id=10)...
  [已缓存] cache_wc2022_teamstats_10.json (11 条记录)
  [请求 11/100] GET /teams/statistics -> 11 条

  采集 Japan (id=12)...
  [已缓存] cache_wc2022_teamstats_12.json (11 条记录)
  [请求 12/100] GET /teams/statistics -> 11 条

  采集 Senegal (id=13)...
  [已缓存] cache_wc2022_teamstats_13.json (11 条记录)
  [请求 13/100] 

In [15]:
# === 3.3 补全 2022 WC 全部球员数据 ===
# 已有: wc2022_top8_players.json (部分球队)
# 对每支 2022 参赛队获取球员数据

existing_players = load_data('wc2022_top8_players.json') or {}
all_wc2022_players = dict(existing_players)

for tid in sorted(wc2022_team_ids.keys()):
    team_name = wc2022_team_ids[tid]
    cache_file = f"cache_wc2022_players_{tid}.json"
    
    if team_name in all_wc2022_players and len(all_wc2022_players[team_name]) > 0:
        print(f"  [已有] {team_name}: {len(all_wc2022_players[team_name])} 球员")
        continue
    
    cached = load_data(cache_file)
    if cached:
        all_wc2022_players[team_name] = cached
        print(f"  [缓存] {team_name}: {len(cached)} 球员")
        continue
    
    if not check_quota():
        print(f"\n[!] 配额不足，停止。")
        break
    
    result = api_get('players', params={'team': tid, 'season': 2022}, cache_file=cache_file)
    if result:
        all_wc2022_players[team_name] = result
        print(f"  {team_name}: {len(result)} 球员")
    else:
        all_wc2022_players[team_name] = []

save_data('wc2022_all_players.json', all_wc2022_players)
total_players = sum(len(p) for p in all_wc2022_players.values())
print(f"\n=== 2022 WC 球员数据: {len(all_wc2022_players)} 队, {total_players} 名球员 ===")

  [保存] wc2022_all_players.json

=== 2022 WC 球员数据: 0 队, 0 名球员 ===


## 4. 2018 世界杯历史数据

采集 2018 世界杯全部比赛、积分榜和球队统计，作为模型训练的历史数据。

> **预估请求**：~40 次（比赛 2 + 积分榜 1 + 球队统计 ~32 + 球员 ~5）

In [16]:
# === 4.1 2018 WC 比赛和积分榜 ===
reset_counter()  # 新天重置计数

# 2018 WC 全部比赛
wc2018_fixtures = api_get_all_pages(
    'fixtures', params={'league': 1, 'season': 2018},
    cache_file='wc2018_fixtures'
)
print(f"\n2018 WC 比赛: {len(wc2018_fixtures)} 场")

# 2018 WC 积分榜
wc2018_standings = api_get(
    'standings', params={'league': 1, 'season': 2018},
    cache_file='wc2018_standings.json'
)
print(f"2018 WC 积分榜: {len(wc2018_standings) if wc2018_standings else 0} 组")

# 2018 WC 参赛队
wc2018_teams = api_get_all_pages(
    'teams', params={'league': 1, 'season': 2018},
    cache_file='cache_wc2018_teams'
)
print(f"2018 WC 参赛队: {len(wc2018_teams)} 支")
wc2018_team_ids = {t['team']['id']: t['team']['name'] for t in wc2018_teams}

print(f"\n剩余配额: {get_remaining()}")

  [API 错误] fixtures: Free plans do not have access to this season, try from 2022 to 2024.
  [重试] 不带 page 参数请求 fixtures...
  [API 错误] fixtures: Free plans do not have access to this season, try from 2022 to 2024.


TypeError: 'NoneType' object is not iterable

In [17]:
# === 4.2 2018 WC 球队统计（仅采集与 2026 重叠的队）===
# 优先采集 2026 参赛队 + 2022 强队
priority_teams_2018 = set()
# 从已有 ID 映射中找出也在 2018 参赛的队
for name, tid in team_id_map.items():
    if tid in wc2018_team_ids:
        priority_teams_2018.add(tid)

print(f"2018 & 2026 重叠球队: {len(priority_teams_2018)} 支")
print(f"剩余配额: {get_remaining()}")

wc2018_stats = {}
for tid in sorted(priority_teams_2018):
    team_name = wc2018_team_ids.get(tid, str(tid))
    cache_file = f"cache_wc2018_teamstats_{tid}.json"
    
    cached = load_data(cache_file)
    if cached:
        wc2018_stats[tid] = cached
        continue
    
    if not check_quota():
        print(f"\n[!] 配额不足，停止。")
        break
    
    result = api_get('teams/statistics', params={
        'league': 1, 'season': 2018, 'team': tid
    }, cache_file=cache_file)
    
    if result:
        wc2018_stats[tid] = result
        print(f"  {team_name}: OK")
    else:
        print(f"  {team_name}: 无数据")

save_data('wc2018_team_stats.json', wc2018_stats)
print(f"\n=== 2018 WC 球队统计: {len(wc2018_stats)} 支 ===")

NameError: name 'wc2018_team_ids' is not defined

## 5. 近期大赛数据（2024-2025）

采集欧洲杯、美洲杯、非洲杯、亚洲杯等近期国际大赛数据，这些是评估球队当前实力的重要依据。

> **预估请求**：~30 次（赛事查找 6 + 比赛/统计 ~24）

In [18]:
# === 5.1 查找各大洲大赛的 league ID ===
reset_counter()

TOURNAMENTS = {
    'Euro': {'search': 'Euro', 'season': 2024},
    'Copa America': {'search': 'Copa America', 'season': 2024},
    'Africa Cup of Nations': {'search': 'Africa', 'season': 2023},
    'Asian Cup': {'search': 'Asian Cup', 'season': 2023},
    'CONCACAF Gold Cup': {'search': 'Gold Cup', 'season': 2023},
    'UEFA Nations League': {'search': 'Nations League', 'season': 2024},
}

tournament_ids = {}

for t_name, config in TOURNAMENTS.items():
    result = api_get('leagues', params={'search': config['search']}, cache_file=f"cache_league_search_{t_name.replace(' ', '_')}.json")
    if result:
        # 找到匹配的赛事
        for league in result:
            l = league['league']
            # 检查赛季是否匹配
            seasons = [s['year'] for s in league.get('seasons', [])]
            if config['season'] in seasons:
                print(f"  {t_name}: id={l['id']}, name={l['name']}, seasons={seasons[:5]}")
                tournament_ids[t_name] = {
                    'id': l['id'],
                    'name': l['name'],
                    'season': config['season']
                }
                break

save_data('tournament_ids.json', tournament_ids)
print(f"\n找到 {len(tournament_ids)} 个赛事")
print(f"剩余配额: {get_remaining()}")

  [缓存命中] cache_league_search_Euro.json (更新于 03-26 20:08)
  Euro: id=4, name=Euro Championship, seasons=[2008, 2012, 2016, 2020, 2024]
  [缓存命中] cache_league_search_Copa_America.json (更新于 03-26 20:08)
  Copa America: id=9, name=Copa America, seasons=[2015, 2016, 2019, 2021, 2024]
  [缓存命中] cache_league_search_Africa_Cup_of_Nations.json (更新于 03-26 20:08)
  Africa Cup of Nations: id=6, name=Africa Cup of Nations, seasons=[2015, 2017, 2019, 2021, 2023]
  [缓存命中] cache_league_search_Asian_Cup.json (更新于 03-26 20:08)
  Asian Cup: id=7, name=Asian Cup, seasons=[2011, 2015, 2019, 2023]
  [缓存命中] cache_league_search_CONCACAF_Gold_Cup.json (更新于 03-26 20:08)
  CONCACAF Gold Cup: id=22, name=CONCACAF Gold Cup, seasons=[2015, 2017, 2019, 2021, 2023]
  [缓存命中] cache_league_search_UEFA_Nations_League.json (更新于 03-26 20:08)
  UEFA Nations League: id=5, name=UEFA Nations League, seasons=[2018, 2020, 2022, 2024, 2026]
  [保存] tournament_ids.json

找到 6 个赛事
剩余配额: 100


In [28]:
# === 5.2 采集各大赛比赛数据和积分榜 ===
tournament_fixtures = {}
tournament_standings = {}

for t_name, t_info in tournament_ids.items():
    lid = t_info['id']
    season = t_info['season']
    safe_name = t_name.replace(' ', '_').replace('(', '').replace(')', '')
    
    print(f"\n{'='*40}")
    print(f"采集: {t_name} (id={lid}, season={season})")
    
    # 比赛
    fixtures = api_get_all_pages(
        'fixtures', params={'league': lid, 'season': season},
        cache_file=f"cache_{safe_name}_{season}_fixtures"
    )
    if fixtures:
        tournament_fixtures[t_name] = fixtures
        print(f"  比赛: {len(fixtures)} 场")
    
    # 积分榜
    standings = api_get(
        'standings', params={'league': lid, 'season': season},
        cache_file=f"{safe_name}_{season}_standings.json"
    )
    if standings:
        tournament_standings[t_name] = standings
        print(f"  积分榜: {len(standings)} 组")
    
    if not check_quota(5):
        print(f"[!] 配额不足，停止采集大赛数据")
        break

# 保存
save_data('tournament_fixtures.json', tournament_fixtures)
save_data('tournament_standings.json', tournament_standings)

total_matches = sum(len(f) for f in tournament_fixtures.values())
print(f"\n=== 大赛数据汇总 ===")
for t_name, fx in tournament_fixtures.items():
    print(f"  {t_name}: {len(fx)} 场比赛")
print(f"总计: {total_matches} 场")


采集: Euro (id=4, season=2024)
  [缓存] Page 1: 51 条
  [API 错误] fixtures: The Page field do not exist.
  [合并] 共 51 条 -> cache_Euro_2024_fixtures_all.json
  比赛: 51 场
  [缓存命中] Euro_2024_standings.json (更新于 03-26 20:09)
  积分榜: 1 组

采集: Copa America (id=9, season=2024)
  [缓存] Page 1: 32 条
  [API 错误] fixtures: The Page field do not exist.
  [合并] 共 32 条 -> cache_Copa_America_2024_fixtures_all.json
  比赛: 32 场
  [缓存命中] Copa_America_2024_standings.json (更新于 03-26 20:09)
  积分榜: 1 组

采集: Africa Cup of Nations (id=6, season=2023)
  [缓存] Page 1: 52 条
  [API 错误] fixtures: The Page field do not exist.
  [合并] 共 52 条 -> cache_Africa_Cup_of_Nations_2023_fixtures_all.json
  比赛: 52 场
  [缓存命中] Africa_Cup_of_Nations_2023_standings.json (更新于 03-26 20:09)
  积分榜: 1 组

采集: Asian Cup (id=7, season=2023)
  [缓存] Page 1: 51 条
  [API 错误] fixtures: The Page field do not exist.
  [合并] 共 51 条 -> cache_Asian_Cup_2023_fixtures_all.json
  比赛: 51 场
  [缓存命中] Asian_Cup_2023_standings.json (更新于 03-26 20:12)
  积分榜: 1 组

采集: CONCA

## 6. 2026 参赛队近期比赛全面采集

对全部 48 支 2026 参赛队采集 2024 和 2023 赛季的国际比赛数据。

> **预估请求**：~96 次（48 队 × 2 个赛季）
> **建议分批**：每次运行设置 `skip_to` 和 `max_teams` 控制数量

## 5.5 补充：生成 2026 WC 参赛队 team_*.json

从 API 获取 2026 世界杯全部参赛队信息，生成 `team_*.json` 文件，供后续 Cell 使用。

In [ ]:
# === 5.5 生成 2026 WC 参赛队 team_*.json ===
# 免费版 API 不支持 2026 赛季，从 Wikipedia 分组信息 + 已有 team_ids 生成本地 team_*.json
# 此 Cell 已预执行，48 支队的 team_*.json 已在 wc2026_raw_data/ 中

import json
from pathlib import Path

print("检查 team_*.json 文件...")
team_files = sorted(DATA_DIR.glob('team_*.json'))
print(f"共 {len(team_files)} 个 team_*.json")

if len(team_files) > 0:
    teams_2026_check = {}
    groups_check = {}
    for f in team_files:
        data = load_data(f.name)
        if data and data.get('team'):
            name = data['team']['name']
            teams_2026_check[name] = data['team']
            group = 'Unknown'
            if data.get('matches'):
                group = data['matches'][0].get('group', 'Unknown')
                if group not in groups_check:
                    groups_check[group] = []
                groups_check[group].append(name)

    print(f"\n2026 参赛队: {len(teams_2026_check)} 支")
    print(f"\n=== 分组 ===")
    for g in sorted(groups_check.keys()):
        print(f"  {g}: {', '.join(sorted(groups_check[g]))}")
else:
    print("[!] 未找到 team_*.json，需要先运行分组数据生成")

In [29]:
# === 6.1 全部 2026 参赛队近期比赛 ===
reset_counter()

# 构建 2026 队名 -> API-Sports ID 映射（反向）
name_to_id = {}
for name, tid in team_id_map.items():
    if name in teams_2026:
        name_to_id[name] = tid

print(f"已匹配 ID 的 2026 参赛队: {len(name_to_id)} 支")
print(f"剩余配额: {get_remaining()}")

# ============ 分批控制 ============
skip_to = 0        # 从第几支队开始
max_teams = 50     # 本次最多采集几支（每次 ~25 支 ≈ 50 次请求）
# ================================

all_recent_matches = load_data('all_teams_recent_matches.json') or {}

team_list = sorted(name_to_id.items())
batch = team_list[skip_to : skip_to + max_teams]

for i, (name, tid) in enumerate(batch):
    print(f"\n--- [{skip_to + i + 1}/{len(team_list)}] {name} (id={tid}) ---")
    
    for season in [2024, 2023]:
        cache_file = f"cache_recent_{tid}_{season}.json"
        
        # 检查是否已有缓存
        cached = load_data(cache_file)
        if cached:
            print(f"  [{season}] 缓存: {len(cached)} 场")
            if name not in all_recent_matches:
                all_recent_matches[name] = []
            all_recent_matches[name].extend(cached)
            continue
        
        if not check_quota():
            break
        
        result = api_get_all_pages(
            'fixtures', params={'team': tid, 'season': season},
            cache_file=f"cache_recent_{tid}_{season}",
            max_pages=3  # 限制翻页，国家队比赛通常不多
        )
        
        if result:
            if name not in all_recent_matches:
                all_recent_matches[name] = []
            all_recent_matches[name].extend(result)
            print(f"  [{season}] 获取: {len(result)} 场")
    
    if not check_quota(2):
        print(f"\n[!] 配额不足，下次从 skip_to={skip_to + i + 1} 继续")
        break

# 去重（同一比赛可能出现两次）
for name in all_recent_matches:
    seen = set()
    unique = []
    for m in all_recent_matches[name]:
        mid = m['fixture']['id']
        if mid not in seen:
            seen.add(mid)
            unique.append(m)
    all_recent_matches[name] = unique

save_data('all_teams_recent_matches.json', all_recent_matches)

total = sum(len(m) for m in all_recent_matches.values())
print(f"\n=== 近期比赛汇总: {len(all_recent_matches)} 队, {total} 场 ===")
for name, matches in sorted(all_recent_matches.items(), key=lambda x: -len(x[1])):
    print(f"  {name}: {len(matches)} 场")

已匹配 ID 的 2026 参赛队: 0 支
剩余配额: 100
  [保存] all_teams_recent_matches.json

=== 近期比赛汇总: 0 队, 0 场 ===


## 7. 球员数据采集

对全部 2026 参赛队采集 2024 赛季球员数据。

> **预估请求**：~48 次

In [30]:
# === 7.1 全部 2026 参赛队球员数据 ===
reset_counter()

# ============ 分批控制 ============
skip_to = 0
max_teams = 50
# ================================

all_players_2024 = load_data('all_teams_players_2024.json') or {}

team_list = sorted(name_to_id.items())
batch = team_list[skip_to : skip_to + max_teams]

for i, (name, tid) in enumerate(batch):
    cache_file = f"cache_players_2024_{tid}.json"
    
    if name in all_players_2024 and len(all_players_2024[name]) > 0:
        print(f"  [{skip_to + i + 1}] [已有] {name}: {len(all_players_2024[name])} 球员")
        continue
    
    cached = load_data(cache_file)
    if cached:
        all_players_2024[name] = cached
        print(f"  [{skip_to + i + 1}] [缓存] {name}: {len(cached)} 球员")
        continue
    
    if not check_quota():
        print(f"\n[!] 配额不足，下次从 skip_to={skip_to + i + 1} 继续")
        break
    
    result = api_get_all_pages(
        'players', params={'team': tid, 'season': 2024},
        cache_file=f"cache_players_2024_{tid}",
        max_pages=3
    )
    
    if result:
        all_players_2024[name] = result
        print(f"  [{skip_to + i + 1}] {name}: {len(result)} 球员")
    else:
        all_players_2024[name] = []
        print(f"  [{skip_to + i + 1}] {name}: 无球员数据")

save_data('all_teams_players_2024.json', all_players_2024)
total = sum(len(p) for p in all_players_2024.values())
print(f"\n=== 2024 球员数据: {len(all_players_2024)} 队, {total} 名球员 ===")

  [保存] all_teams_players_2024.json

=== 2024 球员数据: 0 队, 0 名球员 ===


## 8. H2H 交锋矩阵

采集所有小组内对手之间的历史交锋记录，以及强队之间的 H2H 数据。

> **预估请求**：~72 次（12 组 × 6 对 = 72 对）
> **建议分批**：每次运行设置 `skip_to`

In [22]:
# === 8.1 生成所有需要采集的 H2H 对阵 ===
reset_counter()

# 小组内对阵
h2h_pairs = []
for group_name, team_names in groups.items():
    team_names_sorted = sorted(team_names)
    for i in range(len(team_names_sorted)):
        for j in range(i + 1, len(team_names_sorted)):
            t1, t2 = team_names_sorted[i], team_names_sorted[j]
            id1 = name_to_id.get(t1)
            id2 = name_to_id.get(t2)
            if id1 and id2:
                h2h_pairs.append((t1, t2, id1, id2, f"{group_name}"))

# 额外：传统强队之间的 H2H（跨组）
top_teams = ['Argentina', 'Brazil', 'France', 'England', 'Spain', 'Germany', 
             'Portugal', 'Netherlands', 'Croatia', 'Italy', 'Belgium', 'Uruguay']
for i in range(len(top_teams)):
    for j in range(i + 1, len(top_teams)):
        t1, t2 = top_teams[i], top_teams[j]
        id1 = name_to_id.get(t1)
        id2 = name_to_id.get(t2)
        if id1 and id2:
            pair_key = tuple(sorted([t1, t2]))
            # 避免重复（如果已在小组内采集过）
            if not any(tuple(sorted([p[0], p[1]])) == pair_key for p in h2h_pairs):
                h2h_pairs.append((t1, t2, id1, id2, "top_teams"))

print(f"H2H 对阵总数: {len(h2h_pairs)}")
print(f"剩余配额: {get_remaining()}")

# ============ 分批控制 ============
skip_to = 0
max_pairs = 80
# ================================

batch = h2h_pairs[skip_to : skip_to + max_pairs]
h2h_data = load_data('all_h2h_data.json') or {}

for i, (t1, t2, id1, id2, source) in enumerate(batch):
    pair_key = f"{t1}-{t2}"
    pair_key_rev = f"{t2}-{t1}"
    
    if pair_key in h2h_data or pair_key_rev in h2h_data:
        continue
    
    if not check_quota():
        print(f"\n[!] 配额不足，下次从 skip_to={skip_to + i + 1} 继续")
        break
    
    cache_file = f"cache_h2h_{id1}_{id2}.json"
    cached = load_data(cache_file)
    
    if cached:
        h2h_data[pair_key] = cached
        print(f"  [{skip_to + i + 1}] [缓存] {t1} vs {t2}: {len(cached)} 场")
        continue
    
    print(f"  [{skip_to + i + 1}] 采集 {t1} vs {t2} ({source})...")
    result = api_get('fixtures/headtohead', params={'h2h': f"{id1}-{id2}"}, cache_file=cache_file)
    
    if result:
        h2h_data[pair_key] = result
        print(f"    -> {len(result)} 场历史交锋")

save_data('all_h2h_data.json', h2h_data)
print(f"\n=== H2H 数据: {len(h2h_data)} 组对阵 ===")

# 统计
total_matches = sum(len(m) for m in h2h_data.values())
print(f"总交锋记录: {total_matches} 场")
for key in sorted(h2h_data.keys()):
    print(f"  {key}: {len(h2h_data[key])} 场")

H2H 对阵总数: 0
剩余配额: 100
  [保存] all_h2h_data.json

=== H2H 数据: 0 组对阵 ===
总交锋记录: 0 场


## 9. 2026 世界杯赛程数据

采集 2026 世界杯赛程、参赛球队和实时积分榜（比赛开始后可用）。

> **预估请求**：~5 次

In [23]:
# === 9.1 2026 WC 赛程和参赛队 ===

# 2026 WC 全部赛程
wc2026_fixtures = api_get_all_pages(
    'fixtures', params={'league': 1, 'season': 2026},
    cache_file='wc2026_api_fixtures'
)
print(f"2026 WC 赛程: {len(wc2026_fixtures)} 场")

# 2026 WC 参赛队确认
wc2026_teams_api = api_get_all_pages(
    'teams', params={'league': 1, 'season': 2026},
    cache_file='cache_wc2026_teams_api'
)
print(f"2026 WC 参赛队 (API): {len(wc2026_teams_api)} 支")
for t in wc2026_teams_api:
    print(f"  {t['team']['name']} (id={t['team']['id']})")

# 2026 WC 积分榜（比赛开始后才可用）
wc2026_standings = api_get(
    'standings', params={'league': 1, 'season': 2026},
    cache_file='wc2026_standings.json'
)
if wc2026_standings:
    print(f"2026 WC 积分榜: {len(wc2026_standings)} 组")
else:
    print("2026 WC 积分榜: 尚无数据（比赛未开始）")

# 保存
if wc2026_fixtures:
    save_data('wc2026_api_fixtures.json', wc2026_fixtures)
if wc2026_teams_api:
    save_data('wc2026_api_teams.json', {t['team']['id']: t['team']['name'] for t in wc2026_teams_api})

  [API 错误] fixtures: Free plans do not have access to this season, try from 2022 to 2024.
  [重试] 不带 page 参数请求 fixtures...
  [API 错误] fixtures: Free plans do not have access to this season, try from 2022 to 2024.


TypeError: 'NoneType' object is not iterable

## 10. 数据完整性检查与汇总

统计所有已采集数据，检查缺失项，生成下一步采集计划。

In [ ]:
# === 10.1 数据完整性检查 ===
import pandas as pd

print("=" * 60)
print("数据完整性检查报告")
print("=" * 60)

# 检查各类数据
checks = {
    '2026 WC 赛程 (API)': ('wc2026_api_fixtures.json', lambda d: f"{len(d)} 场" if d else "未采集"),
    '2026 WC 参赛队 (API)': ('wc2026_api_teams.json', lambda d: f"{len(d)} 支" if d else "未采集"),
    '2026 WC 分组 (本地)': ('team_*.json', lambda d: f"{d} 支"),
    '2022 WC 比赛': ('wc2022_fixtures.json', lambda d: f"{len(d)} 场" if d else "未采集"),
    '2022 WC 积分榜': ('wc2022_standings.json', lambda d: f"{len(d)} 组" if d else "未采集"),
    '2022 WC 球队统计 (全部32队)': ('wc2022_all_team_stats.json', lambda d: f"{len(d)}/32 队" if d else "未采集"),
    '2022 WC 球员数据': ('wc2022_all_players.json', lambda d: f"{len(d)} 队" if d else "未采集"),
    '2018 WC 比赛': ('wc2018_fixtures_all.json', lambda d: f"{len(d)} 场" if d else "未采集"),
    '2018 WC 积分榜': ('wc2018_standings.json', lambda d: f"{len(d)} 组" if d else "未采集"),
    '2018 WC 球队统计': ('wc2018_team_stats.json', lambda d: f"{len(d)} 队" if d else "未采集"),
    '近期比赛 (全部队)': ('all_teams_recent_matches.json', lambda d: f"{len(d)} 队, {sum(len(m) for m in d.values())} 场" if d else "未采集"),
    '球员数据 2024': ('all_teams_players_2024.json', lambda d: f"{len(d)} 队, {sum(len(p) for p in d.values())} 人" if d else "未采集"),
    'H2H 对阵': ('all_h2h_data.json', lambda d: f"{len(d)} 组, {sum(len(m) for m in d.values())} 场" if d else "未采集"),
    '大赛比赛数据': ('tournament_fixtures.json', lambda d: f"{len(d)} 个赛事" if d else "未采集"),
    '大赛积分榜': ('tournament_standings.json', lambda d: f"{len(d)} 个赛事" if d else "未采集"),
    '球队 ID 映射': ('../api_sports_team_ids.json', lambda d: f"{len(d)} 支" if d else "未采集"),
    '淘汰赛预测': ('wc2022_knockout_predictions.json', lambda d: f"{len(d)} 场" if d else "未采集"),
}

print(f"\n{'数据项':<35} {'状态':<20}")
print("-" * 55)

for name, (filename, fmt) in checks.items():
    if '*' in filename:
        count = len(list(DATA_DIR.glob(filename)))
        status = fmt(count)
    else:
        data = load_data(filename)
        status = fmt(data)
    print(f"  {name:<33} {status}")

数据完整性检查报告

数据项                                 状态                  
-------------------------------------------------------
  2026 WC 赛程 (API)                  未采集
  2026 WC 参赛队 (API)                 未采集
  2026 WC 分组 (本地)                   0 支
  2022 WC 比赛                        未采集
  2022 WC 积分榜                       未采集
  2022 WC 球队统计 (全部32队)              未采集
  2022 WC 球员数据                      未采集
  2018 WC 比赛                        未采集
  2018 WC 积分榜                       未采集
  2018 WC 球队统计                      未采集
  近期比赛 (全部队)                        未采集
  球员数据 2024                         未采集
  H2H 对阵                            未采集
  大赛比赛数据                            6 个赛事
  大赛积分榜                             6 个赛事
  球队 ID 映射                          49 支
  淘汰赛预测                             未采集


In [ ]:
# === 10.2 每支球队的数据覆盖度 ===
print("\n" + "=" * 60)
print("每支球队数据覆盖度")
print("=" * 60)

# 加载所有数据
recent = load_data('all_teams_recent_matches.json') or {}
players_2024 = load_data('all_teams_players_2024.json') or {}
wc2022_stats_full = load_data('wc2022_all_team_stats.json') or {}
wc2018_stats_full = load_data('wc2018_team_stats.json') or {}

# 按 2026 参赛队统计
print(f"\n{'球队':<20} {'近期比赛':>8} {'2024球员':>8} {'2022统计':>8} {'2018统计':>8}")
print("-" * 55)

for name in sorted(teams_2026.keys()):
    recent_count = len(recent.get(name, []))
    player_count = len(players_2024.get(name, []))
    
    # 检查 2022 统计
    tid = name_to_id.get(name)
    has_2022 = "Y" if (tid and str(tid) in {str(k) for k in wc2022_stats_full.keys()}) else "-"
    has_2018 = "Y" if (tid and str(tid) in {str(k) for k in wc2018_stats_full.keys()}) else "-"
    
    flag_2022 = f"[{has_2022}]" if has_2022 != "-" else ""
    flag_2018 = f"[{has_2018}]" if has_2018 != "-" else ""
    
    print(f"  {name:<18} {recent_count:>8} {player_count:>8} {flag_2022:>8} {flag_2018:>8}")


每支球队数据覆盖度

球队                       近期比赛   2024球员   2022统计   2018统计
-------------------------------------------------------


In [ ]:
# === 10.3 生成采集进度日志 ===
from datetime import datetime

log = {
    'timestamp': datetime.now().isoformat(),
    'total_requests_used': request_count,
    'daily_limit': DAILY_LIMIT,
    'files_in_data_dir': len(list(DATA_DIR.glob('*.json'))),
    'data_size_mb': round(sum(f.stat().st_size for f in DATA_DIR.glob('*.json')) / 1024 / 1024, 2),
    'teams_with_id': len(team_id_map),
    'teams_2026_count': len(teams_2026),
}

print(f"\n{'='*40}")
print(f"采集进度日志")
print(f"{'='*40}")
print(f"  时间: {log['timestamp']}")
print(f"  本次请求: {log['total_requests_used']}/{log['daily_limit']}")
print(f"  数据文件: {log['files_in_data_dir']} 个")
print(f"  数据大小: {log['data_size_mb']} MB")
print(f"  球队 ID 映射: {log['teams_with_id']}/{log['teams_2026_count']} 支")

# 缺失项
missing = []
for name in teams_2026:
    issues = []
    if name not in name_to_id:
        issues.append("无ID")
    if not recent.get(name):
        issues.append("无近期比赛")
    if not players_2024.get(name):
        issues.append("无球员数据")
    if issues:
        missing.append((name, issues))

if missing:
    print(f"\n  缺失数据 ({len(missing)} 队):")
    for name, issues in missing:
        print(f"    {name}: {', '.join(issues)}")
else:
    print(f"\n  所有球队数据完整!")

print(f"\n{'='*40}")
print("下一步建议:")
print("  1. 运行 Cell 2 补全缺失的球队 ID (skip_to=调整)")
print("  2. 运行 Cell 6 补全近期比赛 (skip_to=调整)")
print("  3. 运行 Cell 7 补全球员数据 (skip_to=调整)")
print("  4. 运行 Cell 8 补全 H2H 数据 (skip_to=调整)")
print("  5. 数据完整后，运行 test2.ipynb 进行预测建模")
print(f"{'='*40}")


采集进度日志
  时间: 2026-03-26T20:14:54.686425
  本次请求: 0/100
  数据文件: 33 个
  数据大小: 2.31 MB
  球队 ID 映射: 49/0 支

  所有球队数据完整!

下一步建议:
  1. 运行 Cell 2 补全缺失的球队 ID (skip_to=调整)
  2. 运行 Cell 6 补全近期比赛 (skip_to=调整)
  3. 运行 Cell 7 补全球员数据 (skip_to=调整)
  4. 运行 Cell 8 补全 H2H 数据 (skip_to=调整)
  5. 数据完整后，运行 test2.ipynb 进行预测建模
